In [183]:
import torch

x = torch.tensor(5.0, requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [184]:
f.backward()

In [185]:
x.grad

tensor(10.)

In [186]:
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad

In [187]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x ** 2
    f.backward()
    with torch.no_grad():
        x -= learning_rate * x.grad

    x.grad.zero_()

### Real Example

In [188]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
housing

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]], shape=(20640, 8)),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': 

In [189]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=123
)
print(X_train_full.shape, X_test.shape)
print(f'X_train_full is {(X_train_full.shape[0] / (X_train_full.shape[0] + X_test.shape[0]) * 100)}%\
      X_test is {(X_test.shape[0] / (X_train_full.shape[0] + X_test.shape[0]) * 100)}%')

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=123
)
print(X_train.shape, X_valid.shape)
print(f'X_train is {X_train.shape[0] / (X_train.shape[0] + X_valid.shape[0]) * 100}%\
      X_valid is {X_valid.shape[0] / (X_train.shape[0] + X_valid.shape[0]) * 100}%')

(15480, 8) (5160, 8)
X_train_full is 75.0%      X_test is 25.0%
(11610, 8) (3870, 8)
X_train is 75.0%      X_valid is 25.0%


In [190]:
X_train

array([[   3.3804    ,   38.        ,    5.83009709, ...,    3.55825243,
          33.9       , -118.25      ],
       [   1.398     ,   14.        ,    5.65422397, ...,    1.98624754,
          37.51      , -119.99      ],
       [   6.9931    ,   25.        ,   10.17318436, ...,    1.94972067,
          38.36      , -122.26      ],
       ...,
       [  15.0001    ,   52.        ,    8.78061224, ...,    3.51020408,
          34.08      , -118.34      ],
       [   3.125     ,    9.        ,    5.55409219, ...,    3.28222013,
          34.54      , -117.32      ],
       [   2.1935    ,   31.        ,    4.79402985, ...,    3.13134328,
          34.16      , -117.34      ]], shape=(11610, 8))

In [191]:
X_train = torch.FloatTensor(X_train)
X_train

tensor([[   3.3804,   38.0000,    5.8301,  ...,    3.5583,   33.9000,
         -118.2500],
        [   1.3980,   14.0000,    5.6542,  ...,    1.9862,   37.5100,
         -119.9900],
        [   6.9931,   25.0000,   10.1732,  ...,    1.9497,   38.3600,
         -122.2600],
        ...,
        [  15.0001,   52.0000,    8.7806,  ...,    3.5102,   34.0800,
         -118.3400],
        [   3.1250,    9.0000,    5.5541,  ...,    3.2822,   34.5400,
         -117.3200],
        [   2.1935,   31.0000,    4.7940,  ...,    3.1313,   34.1600,
         -117.3400]])

In [192]:
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)

In [193]:
means = X_train.mean(dim=0, keepdim=True)

In [194]:
means

tensor([[ 3.8756e+00,  2.8531e+01,  5.4368e+00,  1.0979e+00,  1.4326e+03,
          3.1159e+00,  3.5631e+01, -1.1957e+02]])

In [195]:
stds = X_train.std(dim=0, keepdim=True)

In [196]:
stds

tensor([[1.9053e+00, 1.2533e+01, 2.6640e+00, 5.2507e-01, 1.1161e+03, 1.2652e+01,
         2.1311e+00, 2.0035e+00]])

In [197]:
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

##### PyTorch expects the targets to have one row per sample, so let's reshape the targets to be column vectors:

In [198]:
y_train = torch.FloatTensor(y_train).view(-1, 1)
y_valid = torch.FloatTensor(y_valid).view(-1, 1)
y_test = torch.FloatTensor(y_test).view(-1, 1)

In [199]:
torch.manual_seed(123)
n_features = X_train.shape[1]
w = torch.rand((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [200]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = (y_pred - y_train).pow(2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f'Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}')

Epoch 1/20, Loss: 7.004919052124023
Epoch 2/20, Loss: 1.0551552772521973
Epoch 3/20, Loss: 0.7151469588279724
Epoch 4/20, Loss: 0.6581979990005493
Epoch 5/20, Loss: 0.634869396686554
Epoch 6/20, Loss: 0.6212378740310669
Epoch 7/20, Loss: 0.6115010380744934
Epoch 8/20, Loss: 0.603682279586792
Epoch 9/20, Loss: 0.5970300436019897
Epoch 10/20, Loss: 0.5912222266197205
Epoch 11/20, Loss: 0.5860950350761414
Epoch 12/20, Loss: 0.5815466642379761
Epoch 13/20, Loss: 0.5775026679039001
Epoch 14/20, Loss: 0.5739028453826904
Epoch 15/20, Loss: 0.5706958770751953
Epoch 16/20, Loss: 0.5678371787071228
Epoch 17/20, Loss: 0.5652875900268555
Epoch 18/20, Loss: 0.5630125999450684
Epoch 19/20, Loss: 0.5609814524650574
Epoch 20/20, Loss: 0.5591670870780945


<i style="font-family: 'italic'; color: #27b034; font-size: 1.3em;">
PyTorch tracks all operations to compute gradients. If we update parameters without <b style='font-size: 0.8em; color: red'>no_grad()</b>, PyTorch will also track those updates, which is wrong because updates should not be part of gradient computation. So we use <b style='font-size: 0.8em; color: red'>no_grad()</b>to prevent tracking parameter updates.
</i>

In [201]:
X_new = X_test[:3]
with torch.no_grad():
    y_pred = X_new @ w + b

In [202]:
y_pred

tensor([[2.3276],
        [1.5378],
        [2.0075]])

<b style="font-size: 1.6em; color: #13b1c6">Linear Regression Using PyTorch's High-Level API</b>

In [203]:
import torch.nn as nn

torch.manual_seed(123)
model = nn.Linear(in_features=n_features, out_features=1)

In [204]:
model.bias

Parameter containing:
tensor([-0.2234], requires_grad=True)

In [205]:
model.weight

Parameter containing:
tensor([[-0.1442,  0.0117, -0.1756,  0.1333, -0.3012,  0.2592, -0.2570, -0.2811]],
       requires_grad=True)

In [206]:
for param in model.parameters():
    print(param)

Parameter containing:
tensor([[-0.1442,  0.0117, -0.1756,  0.1333, -0.3012,  0.2592, -0.2570, -0.2811]],
       requires_grad=True)
Parameter containing:
tensor([-0.2234], requires_grad=True)


In [207]:
model(X_train[:2])

tensor([[ 0.0143],
        [-0.1120]], grad_fn=<AddmmBackward0>)

In [208]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [209]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f'Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}')

In [210]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 7.016122817993164
Epoch 2/20, Loss: 0.9121877551078796
Epoch 3/20, Loss: 0.6198614835739136
Epoch 4/20, Loss: 0.5906364321708679
Epoch 5/20, Loss: 0.5816692113876343
Epoch 6/20, Loss: 0.5763687491416931
Epoch 7/20, Loss: 0.5722827911376953
Epoch 8/20, Loss: 0.5688230991363525
Epoch 9/20, Loss: 0.5658059120178223
Epoch 10/20, Loss: 0.563151478767395
Epoch 11/20, Loss: 0.5608097910881042
Epoch 12/20, Loss: 0.5587425231933594
Epoch 13/20, Loss: 0.5569167137145996
Epoch 14/20, Loss: 0.5553040504455566
Epoch 15/20, Loss: 0.5538795590400696
Epoch 16/20, Loss: 0.5526210069656372
Epoch 17/20, Loss: 0.5515090823173523
Epoch 18/20, Loss: 0.5505265593528748
Epoch 19/20, Loss: 0.5496582984924316
Epoch 20/20, Loss: 0.5488908290863037


In [211]:
with torch.no_grad():
    y_pred = model(X_test)

y_pred

tensor([[2.3545],
        [1.5037],
        [2.0259],
        ...,
        [1.6541],
        [1.5987],
        [0.8515]])

In [212]:
print(y_pred[:5])
print(y_test[:5])

tensor([[2.3545],
        [1.5037],
        [2.0259],
        [1.5859],
        [2.6425]])
tensor([[1.5160],
        [0.9920],
        [1.3450],
        [2.3170],
        [4.6290]])


<b style='color: #06ac17; font-size: 1.5em'>
    Implementing a Regression MLP <i style='color: #325bd5; font-size: 0.8em'>(Multi-Layer Perceptron)</i>
</b>

In [213]:
torch.manual_seed(123)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

In [214]:
print(type(model))

<class 'torch.nn.modules.container.Sequential'>


In [215]:
learning_rate = 0.01
optimizer = torch.optim.SGD(params=model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model=model, optimizer=optimizer, criterion=mse, X_train=X_train, y_train=y_train, n_epochs=n_epochs)

Epoch 1/20, Loss: 4.970627307891846
Epoch 2/20, Loss: 4.673458099365234
Epoch 3/20, Loss: 4.398462295532227
Epoch 4/20, Loss: 4.141510486602783
Epoch 5/20, Loss: 3.899442434310913
Epoch 6/20, Loss: 3.6702630519866943
Epoch 7/20, Loss: 3.452765703201294
Epoch 8/20, Loss: 3.2462100982666016
Epoch 9/20, Loss: 3.0507192611694336
Epoch 10/20, Loss: 2.8664052486419678
Epoch 11/20, Loss: 2.6934328079223633
Epoch 12/20, Loss: 2.531843423843384
Epoch 13/20, Loss: 2.3817081451416016
Epoch 14/20, Loss: 2.243095636367798
Epoch 15/20, Loss: 2.1159512996673584
Epoch 16/20, Loss: 2.0001511573791504
Epoch 17/20, Loss: 1.8953619003295898
Epoch 18/20, Loss: 1.8011207580566406
Epoch 19/20, Loss: 1.7168155908584595
Epoch 20/20, Loss: 1.641762375831604


<b style='color: #c40ad1; font-size: 1.5em'>
    Implementing Mini-Batch Gradient Descent using DataLoaders
</b>

In [216]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [217]:
torch.manual_seed(123)
model = nn.Sequential(
    nn.Linear(n_features, 50), nn.ReLU(),
    nn.Linear(50, 40), nn.ReLU(),
    nn.Linear(40, 1)
)

learning_rate = 0.02
optimizer = torch.optim.SGD(params=model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [218]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
        mean_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}')

In [219]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 0.7526
Epoch 2/20, Loss: 0.4489
Epoch 3/20, Loss: 0.4199
Epoch 4/20, Loss: 0.4023
Epoch 5/20, Loss: 0.3922
Epoch 6/20, Loss: 0.3809
Epoch 7/20, Loss: 0.3746
Epoch 8/20, Loss: 0.3745
Epoch 9/20, Loss: 0.3668
Epoch 10/20, Loss: 0.3602
Epoch 11/20, Loss: 0.3547
Epoch 12/20, Loss: 0.3508
Epoch 13/20, Loss: 0.3491
Epoch 14/20, Loss: 0.3443
Epoch 15/20, Loss: 0.3391
Epoch 16/20, Loss: 0.3374
Epoch 17/20, Loss: 0.3334
Epoch 18/20, Loss: 0.3360
Epoch 19/20, Loss: 0.3311
Epoch 20/20, Loss: 0.3286


<b style='font-size: 1.4em; color: #9d0ddb'>
    Model Evaluation
</b>

In [220]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))